# SASRec Time-Aware BPI2012 Colab Train 01

Colab notebook for the first time-aware SASRec experiment on BPI 2012.

Goals:
- reuse the existing `refine_v3_ml50_do025` baseline results instead of retraining them
- train only the new time-aware runs with time-delta bucket embeddings
- compare baseline vs `8-bucket` vs `9-bucket` under both `NDCG@10` and `NDCG@5` model-selection criteria


In [1]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))


torch version: 2.10.0+cu128
cuda available: True
gpu name: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
GITHUB_USERNAME = 'hwbuzz'

DRIVE_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction'
REPO_DIR = '/content/time-aware-behavior-prediction'

DATA_DIR = f'{DRIVE_ROOT}/data/processed/bpi2012_complete_only'
BASELINE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012'
BASELINE_NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5'
TIMEAWARE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012'
TIMEAWARE_NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012_ndcg5'
NOTEBOOK_DIR = f'{DRIVE_ROOT}/notebooks'

print('DATA_DIR:', DATA_DIR)
print('BASELINE_NDCG10_OUTPUT_DIR:', BASELINE_NDCG10_OUTPUT_DIR)
print('BASELINE_NDCG5_OUTPUT_DIR:', BASELINE_NDCG5_OUTPUT_DIR)
print('TIMEAWARE_NDCG10_OUTPUT_DIR:', TIMEAWARE_NDCG10_OUTPUT_DIR)
print('TIMEAWARE_NDCG5_OUTPUT_DIR:', TIMEAWARE_NDCG5_OUTPUT_DIR)


DATA_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only
BASELINE_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012
BASELINE_NDCG5_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5
TIMEAWARE_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012
TIMEAWARE_NDCG5_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012_ndcg5


In [4]:
!mkdir -p "$NOTEBOOK_DIR"
!mkdir -p "$DATA_DIR"
!mkdir -p "$BASELINE_NDCG10_OUTPUT_DIR"
!mkdir -p "$BASELINE_NDCG5_OUTPUT_DIR"
!mkdir -p "$TIMEAWARE_NDCG10_OUTPUT_DIR"
!mkdir -p "$TIMEAWARE_NDCG5_OUTPUT_DIR"


In [5]:
%cd /content
!test -d time-aware-behavior-prediction || git clone https://github.com/$GITHUB_USERNAME/time-aware-behavior-prediction.git
%cd /content/time-aware-behavior-prediction


/content
/content/time-aware-behavior-prediction


In [6]:
# If you need the latest code from GitHub, uncomment below.
# %cd /content/time-aware-behavior-prediction
# !git pull


In [7]:
%cd /content/time-aware-behavior-prediction

skip_packages = ['pywinpty']

with open('requirements.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open('requirements_colab.txt', 'w', encoding='utf-8') as f:
    for line in lines:
        pkg = line.strip().lower()
        if not any(name in pkg for name in skip_packages):
            f.write(line)

print('created requirements_colab.txt')


/content/time-aware-behavior-prediction
created requirements_colab.txt


In [8]:
!pip install -r requirements_colab.txt


In [9]:
!ls "$DATA_DIR"


events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


In [10]:
%cd /content/time-aware-behavior-prediction
!mkdir -p data/processed
!cp -r "$DATA_DIR" data/processed/
!ls data/processed/bpi2012_complete_only


/content/time-aware-behavior-prediction
events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


## Experiment design

Fixed baseline setting:
- `refine_v3_ml50_do025`
- `hidden_units=50, num_blocks=2, num_heads=1, maxlen=50, lr=0.001, dropout=0.25`
- seeds: `42`, `2024`

Comparison targets:
- baseline (reuse existing completed runs)
- time-aware `8-bucket`
- time-aware `9-bucket`

Bucket designs:
- `8-bucket`: `padding`, `first`, `zero-gap`, `(0,1m)`, `[1m,10m)`, `[10m,1h)`, `[1h,1d)`, `[>=1d]`
- `9-bucket`: `padding`, `first`, `zero-gap`, `(0,1m)`, `[1m,10m)`, `[10m,1h)`, `[1h,1d)`, `[1d,7d)`, `[>=7d]`


## Check existing baseline runs

These baseline runs should already exist and must not be retrained.


In [11]:
from pathlib import Path

baseline_ndcg10_runs = [
    'refine_v3_ml50_do025_seed42',
    'refine_v3_ml50_do025_seed2024',
]
baseline_ndcg5_runs = [
    'refine_v3_ml50_do025_seed42_ndcg5',
    'refine_v3_ml50_do025_seed2024_ndcg5',
]

for label, output_dir, run_names in [
    ('Baseline NDCG@10', Path(BASELINE_NDCG10_OUTPUT_DIR), baseline_ndcg10_runs),
    ('Baseline NDCG@5', Path(BASELINE_NDCG5_OUTPUT_DIR), baseline_ndcg5_runs),
]:
    print('=' * 80)
    print(label)
    for run_name in run_names:
        run_dir = output_dir / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'MISSING')


Baseline NDCG@10
refine_v3_ml50_do025_seed42 EXISTS
refine_v3_ml50_do025_seed2024 EXISTS
Baseline NDCG@5
refine_v3_ml50_do025_seed42_ndcg5 EXISTS
refine_v3_ml50_do025_seed2024_ndcg5 EXISTS


## Check planned time-aware runs

Only train runs that are still missing.


In [12]:
planned_ndcg10 = [
    'timeaware_refine_v3_ml50_do025_seed42_b8',
    'timeaware_refine_v3_ml50_do025_seed2024_b8',
    'timeaware_refine_v3_ml50_do025_seed42_b9',
    'timeaware_refine_v3_ml50_do025_seed2024_b9',
]
planned_ndcg5 = [
    'timeaware_refine_v3_ml50_do025_seed42_b8_ndcg5',
    'timeaware_refine_v3_ml50_do025_seed2024_b8_ndcg5',
    'timeaware_refine_v3_ml50_do025_seed42_b9_ndcg5',
    'timeaware_refine_v3_ml50_do025_seed2024_b9_ndcg5',
]

for label, output_dir, run_names in [
    ('Time-aware NDCG@10', Path(TIMEAWARE_NDCG10_OUTPUT_DIR), planned_ndcg10),
    ('Time-aware NDCG@5', Path(TIMEAWARE_NDCG5_OUTPUT_DIR), planned_ndcg5),
]:
    print('=' * 80)
    print(label)
    for run_name in run_names:
        run_dir = output_dir / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'OK')


Time-aware NDCG@10
timeaware_refine_v3_ml50_do025_seed42_b8 EXISTS
timeaware_refine_v3_ml50_do025_seed2024_b8 OK
timeaware_refine_v3_ml50_do025_seed42_b9 OK
timeaware_refine_v3_ml50_do025_seed2024_b9 OK
Time-aware NDCG@5
timeaware_refine_v3_ml50_do025_seed42_b8_ndcg5 OK
timeaware_refine_v3_ml50_do025_seed2024_b8_ndcg5 OK
timeaware_refine_v3_ml50_do025_seed42_b9_ndcg5 OK
timeaware_refine_v3_ml50_do025_seed2024_b9_ndcg5 OK


## Train time-aware runs for `NDCG@10`

Run these cells only if the corresponding run directory does not already exist.


### timeaware_refine_v3_ml50_do025_seed42_b8


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware_refine_v3_ml50_do025_seed42_b8 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.25 \
  --seed 42 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012/timeaware_refine_v3_ml50_do025_seed42_b8
epoch=1, loss=0.5230
epoch=2, loss=0.2295
epoch=3, loss=0.1714
epoch=4, loss=0.1357
epoch=5, loss=0.1166
valid [full], NDCG@5: 0.3808, HR@5: 0.5651, NDCG@10: 0.4552, HR@10: 0.7840, MRR: 0.3650
valid [sampled], NDCG@5: 0.5471, HR@5: 0.5583, NDCG@10: 0.5616, HR@10: 0.6031, MRR: 0.5610
test [full], NDCG@5: 0.2770, HR@5: 0.4411, NDCG@10: 0.3223, HR@10: 0.5745, MRR: 0.2684
test [sampled], NDCG@5: 0.4086, HR@5: 0.4214, NDCG@10: 0.4495, HR@10: 0.5519, MRR: 0.4423
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-awar

### timeaware_refine_v3_ml50_do025_seed2024_b8


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware_refine_v3_ml50_do025_seed2024_b8 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.25 \
  --seed 2024 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012/timeaware01_refine_v3_ml50_do025_seed2024_b8
epoch=1, loss=0.5519
epoch=2, loss=0.2364
epoch=3, loss=0.1674
epoch=4, loss=0.1325
epoch=5, loss=0.1179
valid [full], NDCG@5: 0.3085, HR@5: 0.4196, NDCG@10: 0.4362, HR@10: 0.8359, MRR: 0.3342
valid [sampled], NDCG@5: 0.5429, HR@5: 0.5470, NDCG@10: 0.5507, HR@10: 0.5718, MRR: 0.5610
test [full], NDCG@5: 0.1770, HR@5: 0.3432, NDCG@10: 0.3535, HR@10: 0.8840, MRR: 0.2085
test [sampled], NDCG@5: 0.6434, HR@5: 0.6537, NDCG@10: 0.6671, HR@10: 0.7284, MRR: 0.6627
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-

### timeaware_refine_v3_ml50_do025_seed42_b9


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware_refine_v3_ml50_do025_seed42_b9 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.25 \
  --seed 42 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012/timeaware01_refine_v3_ml50_do025_seed42_b9
epoch=1, loss=0.5209
epoch=2, loss=0.2295
epoch=3, loss=0.1706
epoch=4, loss=0.1350
epoch=5, loss=0.1166
valid [full], NDCG@5: 0.3670, HR@5: 0.5348, NDCG@10: 0.4507, HR@10: 0.7846, MRR: 0.3604
valid [sampled], NDCG@5: 0.5448, HR@5: 0.5564, NDCG@10: 0.5597, HR@10: 0.6023, MRR: 0.5586
test [full], NDCG@5: 0.2480, HR@5: 0.3927, NDCG@10: 0.3059, HR@10: 0.5675, MRR: 0.2506
test [sampled], NDCG@5: 0.3852, HR@5: 0.3956, NDCG@10: 0.4367, HR@10: 0.5608, MRR: 0.4243
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aw

### timeaware_refine_v3_ml50_do025_seed2024_b9


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware_refine_v3_ml50_do025_seed2024_b9 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.25 \
  --seed 2024 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012/timeaware01_refine_v3_ml50_do025_seed2024_b9
epoch=1, loss=0.5482
epoch=2, loss=0.2356
epoch=3, loss=0.1668
epoch=4, loss=0.1322
epoch=5, loss=0.1178
valid [full], NDCG@5: 0.3212, HR@5: 0.4373, NDCG@10: 0.4419, HR@10: 0.8269, MRR: 0.3438
valid [sampled], NDCG@5: 0.5618, HR@5: 0.5680, NDCG@10: 0.5733, HR@10: 0.6042, MRR: 0.5809
test [full], NDCG@5: 0.2013, HR@5: 0.4035, NDCG@10: 0.3591, HR@10: 0.8761, MRR: 0.2163
test [sampled], NDCG@5: 0.6374, HR@5: 0.6534, NDCG@10: 0.6654, HR@10: 0.7416, MRR: 0.6562
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-

## Train time-aware runs for `NDCG@5`

Run these cells only if the corresponding run directory does not already exist.


### timeaware_refine_v3_ml50_do025_seed42_b8_ndcg5


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware_refine_v3_ml50_do025_seed42_b8_ndcg5 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.25 \
  --seed 42 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012_ndcg5/timeaware01_refine_v3_ml50_do025_seed42_b8_ndcg5
epoch=1, loss=0.5230
epoch=2, loss=0.2295
epoch=3, loss=0.1714
epoch=4, loss=0.1357
epoch=5, loss=0.1166
valid [full], NDCG@5: 0.3808, HR@5: 0.5651, NDCG@10: 0.4552, HR@10: 0.7840, MRR: 0.3650
valid [sampled], NDCG@5: 0.5471, HR@5: 0.5583, NDCG@10: 0.5616, HR@10: 0.6031, MRR: 0.5610
test [full], NDCG@5: 0.2770, HR@5: 0.4411, NDCG@10: 0.3223, HR@10: 0.5745, MRR: 0.2684
test [sampled], NDCG@5: 0.4086, HR@5: 0.4214, NDCG@10: 0.4495, HR@10: 0.5519, MRR: 0.4423
saved eval checkpoint: /content/drive/MyDrive/ai-proje

### timeaware_refine_v3_ml50_do025_seed2024_b8_ndcg5


In [13]:
!python src/train_sasrec.py \
  --run_name timeaware_refine_v3_ml50_do025_seed2024_b8_ndcg5 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.25 \
  --seed 2024 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012_ndcg5/timeaware_refine_v3_ml50_do025_seed2024_b8_ndcg5
epoch=1, loss=0.5519
epoch=2, loss=0.2364
epoch=3, loss=0.1674
epoch=4, loss=0.1325
epoch=5, loss=0.1179
valid [full], NDCG@5: 0.3085, HR@5: 0.4196, NDCG@10: 0.4362, HR@10: 0.8359, MRR: 0.3342
valid [sampled], NDCG@5: 0.5429, HR@5: 0.5470, NDCG@10: 0.5507, HR@10: 0.5718, MRR: 0.5610
test [full], NDCG@5: 0.1770, HR@5: 0.3432, NDCG@10: 0.3535, HR@10: 0.8840, MRR: 0.2085
test [sampled], NDCG@5: 0.6434, HR@5: 0.6537, NDCG@10: 0.6671, HR@10: 0.7284, MRR: 0.6627
saved eval checkpoint: /content/drive/MyDrive/ai-proje

### timeaware_refine_v3_ml50_do025_seed42_b9_ndcg5


In [14]:
!python src/train_sasrec.py \
  --run_name timeaware_refine_v3_ml50_do025_seed42_b9_ndcg5 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.25 \
  --seed 42 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012_ndcg5/timeaware_refine_v3_ml50_do025_seed42_b9_ndcg5
epoch=1, loss=0.5209
epoch=2, loss=0.2295
epoch=3, loss=0.1706
epoch=4, loss=0.1350
epoch=5, loss=0.1166
valid [full], NDCG@5: 0.3670, HR@5: 0.5348, NDCG@10: 0.4507, HR@10: 0.7846, MRR: 0.3604
valid [sampled], NDCG@5: 0.5448, HR@5: 0.5564, NDCG@10: 0.5597, HR@10: 0.6023, MRR: 0.5586
test [full], NDCG@5: 0.2480, HR@5: 0.3927, NDCG@10: 0.3059, HR@10: 0.5675, MRR: 0.2506
test [sampled], NDCG@5: 0.3852, HR@5: 0.3956, NDCG@10: 0.4367, HR@10: 0.5608, MRR: 0.4243
saved eval checkpoint: /content/drive/MyDrive/ai-project

### timeaware_refine_v3_ml50_do025_seed2024_b9_ndcg5


In [15]:
!python src/train_sasrec.py \
  --run_name timeaware_refine_v3_ml50_do025_seed2024_b9_ndcg5 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.25 \
  --seed 2024 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012_ndcg5/timeaware_refine_v3_ml50_do025_seed2024_b9_ndcg5
epoch=1, loss=0.5482
epoch=2, loss=0.2356
epoch=3, loss=0.1668
epoch=4, loss=0.1322
epoch=5, loss=0.1178
valid [full], NDCG@5: 0.3212, HR@5: 0.4373, NDCG@10: 0.4419, HR@10: 0.8269, MRR: 0.3438
valid [sampled], NDCG@5: 0.5618, HR@5: 0.5680, NDCG@10: 0.5733, HR@10: 0.6042, MRR: 0.5809
test [full], NDCG@5: 0.2013, HR@5: 0.4035, NDCG@10: 0.3591, HR@10: 0.8761, MRR: 0.2163
test [sampled], NDCG@5: 0.6374, HR@5: 0.6534, NDCG@10: 0.6654, HR@10: 0.7416, MRR: 0.6562
saved eval checkpoint: /content/drive/MyDrive/ai-proje

## Rebuild result tables from run folders

This avoids schema issues in `experiment_index.csv` and lets us combine old baseline runs with new time-aware runs safely.


In [16]:
from pathlib import Path
import json
import pandas as pd

def rebuild_df(output_dir: str):
    rows = []
    output_path = Path(output_dir)
    if not output_path.exists():
        return pd.DataFrame()
    for run_dir in output_path.iterdir():
        if not run_dir.is_dir():
            continue
        summary_path = run_dir / 'metrics_summary.json'
        config_path = run_dir / 'config.json'
        if not summary_path.exists() or not config_path.exists():
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        config = json.loads(config_path.read_text(encoding='utf-8'))
        row = {
            'run_name': summary.get('run_name'),
            'run_dir': str(run_dir),
            'completed_at': summary.get('completed_at'),
            'best_epoch': summary.get('best_epoch'),
            'checkpoint_best': summary.get('checkpoint_best'),
            'checkpoint_last': summary.get('checkpoint_last'),
            'metrics_history': summary.get('metrics_history'),
            'config_path': str(config_path),
            'metrics_summary': str(summary_path),
            'maxlen': config.get('maxlen'),
            'dropout_rate': config.get('dropout_rate'),
            'hidden_units': config.get('hidden_units'),
            'seed': config.get('seed'),
            'selection_metric': config.get('selection_metric'),
            'use_time_embedding': config.get('use_time_embedding', False),
            'time_bucket_boundaries': ','.join(str(x) for x in config.get('time_bucket_boundaries', [])),
            'time_bucket_count': config.get('time_bucket_count'),
        }
        best_valid = summary.get('best_valid', {})
        best_test = summary.get('best_test_at_best_valid', {})
        def pick(metrics_group, mode, key):
            return metrics_group.get(mode, {}).get(key)
        row.update({
            'best_valid_full_ndcg@10': pick(best_valid, 'full', 'ndcg@10'),
            'best_valid_full_hr@10': pick(best_valid, 'full', 'hr@10'),
            'best_valid_full_ndcg@5': pick(best_valid, 'full', 'ndcg@5'),
            'best_valid_full_hr@5': pick(best_valid, 'full', 'hr@5'),
            'best_valid_full_mrr': pick(best_valid, 'full', 'mrr'),
            'best_test_full_ndcg@10': pick(best_test, 'full', 'ndcg@10'),
            'best_test_full_hr@10': pick(best_test, 'full', 'hr@10'),
            'best_test_full_ndcg@5': pick(best_test, 'full', 'ndcg@5'),
            'best_test_full_hr@5': pick(best_test, 'full', 'hr@5'),
            'best_test_full_mrr': pick(best_test, 'full', 'mrr'),
            'best_valid_sampled_ndcg@10': pick(best_valid, 'sampled', 'ndcg@10'),
            'best_valid_sampled_hr@10': pick(best_valid, 'sampled', 'hr@10'),
            'best_valid_sampled_ndcg@5': pick(best_valid, 'sampled', 'ndcg@5'),
            'best_valid_sampled_hr@5': pick(best_valid, 'sampled', 'hr@5'),
            'best_valid_sampled_mrr': pick(best_valid, 'sampled', 'mrr'),
            'best_test_sampled_ndcg@10': pick(best_test, 'sampled', 'ndcg@10'),
            'best_test_sampled_hr@10': pick(best_test, 'sampled', 'hr@10'),
            'best_test_sampled_ndcg@5': pick(best_test, 'sampled', 'ndcg@5'),
            'best_test_sampled_hr@5': pick(best_test, 'sampled', 'hr@5'),
            'best_test_sampled_mrr': pick(best_test, 'sampled', 'mrr'),
        })
        rows.append(row)
    return pd.DataFrame(rows)


## NDCG@10 comparison summary


In [17]:
ndcg10_baseline_runs = [
    'refine_v3_ml50_do025_seed42',
    'refine_v3_ml50_do025_seed2024',
]
ndcg10_timeaware_runs = [
    'timeaware_refine_v3_ml50_do025_seed42_b8',
    'timeaware_refine_v3_ml50_do025_seed2024_b8',
    'timeaware_refine_v3_ml50_do025_seed42_b9',
    'timeaware_refine_v3_ml50_do025_seed2024_b9',
]

baseline_df = rebuild_df(BASELINE_NDCG10_OUTPUT_DIR)
timeaware_df = rebuild_df(TIMEAWARE_NDCG10_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(ndcg10_baseline_runs)].copy()
baseline_subset['model_variant'] = 'baseline'
baseline_subset['bucket_variant'] = 'baseline'

timeaware_subset = timeaware_df[timeaware_df['run_name'].isin(ndcg10_timeaware_runs)].copy()
timeaware_subset['model_variant'] = 'timeaware'
timeaware_subset['bucket_variant'] = timeaware_subset['run_name'].apply(lambda x: 'b8' if '_b8' in x else 'b9')

df_ndcg10 = pd.concat([baseline_subset, timeaware_subset], ignore_index=True)
df_ndcg10 = df_ndcg10.sort_values(['bucket_variant', 'seed', 'run_name']).reset_index(drop=True)
df_ndcg10[[
    'run_name', 'seed', 'bucket_variant', 'use_time_embedding', 'time_bucket_boundaries',
    'best_valid_full_ndcg@5', 'best_test_full_ndcg@5',
    'best_valid_full_ndcg@10', 'best_test_full_ndcg@10',
    'best_valid_full_mrr', 'best_test_full_mrr',
    'best_valid_sampled_ndcg@5', 'best_test_sampled_ndcg@5',
    'best_valid_sampled_ndcg@10', 'best_test_sampled_ndcg@10',
    'best_valid_sampled_mrr', 'best_test_sampled_mrr',
]]


,run_name,seed,bucket_variant,use_time_embedding,time_bucket_boundaries,best_valid_full_ndcg@5,best_test_full_ndcg@5,best_valid_full_ndcg@10,best_test_full_ndcg@10,best_valid_full_mrr,best_test_full_mrr,best_valid_sampled_ndcg@5,best_test_sampled_ndcg@5,best_valid_sampled_ndcg@10,best_test_sampled_ndcg@10,best_valid_sampled_mrr,best_test_sampled_mrr
0,timeaware_refine_v3_ml50_do025_seed42_b8,42,b8,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0",0.393344,0.506900,0.518353,0.549708,0.378457,0.410007,0.542432,0.591895,0.558799,0.614605,0.559129,0.616780
1,refine_v3_ml50_do025_seed42,42,baseline,False,,0.387751,0.416791,0.512117,0.546236,0.372049,0.406292,0.541576,0.809862,0.550634,0.820534,0.557796,0.816182
2,refine_v3_ml50_do025_seed2024,2024,baseline,False,,0.376172,0.337719,0.500475,0.460943,0.356303,0.299817,0.553469,0.811153,0.569319,0.821949,0.569866,0.816608


In [18]:
summary_ndcg10 = df_ndcg10.groupby('bucket_variant')[[
    'best_valid_full_ndcg@5', 'best_test_full_ndcg@5',
    'best_valid_full_ndcg@10', 'best_test_full_ndcg@10',
    'best_valid_full_mrr', 'best_test_full_mrr',
    'best_valid_sampled_ndcg@5', 'best_test_sampled_ndcg@5',
    'best_valid_sampled_ndcg@10', 'best_test_sampled_ndcg@10',
    'best_valid_sampled_mrr', 'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary_ndcg10


best_valid_full_ndcg@5           best_test_full_ndcg@5  \
                                 mean       std                  mean   
bucket_variant                                                          
b8                           0.393344       NaN              0.506900   
baseline                     0.381962  0.008188              0.377255   

                         best_valid_full_ndcg@10            \
                     std                    mean       std   
bucket_variant                                               
b8                   NaN                0.518353       NaN   
baseline        0.055912                0.506296  0.008232   

               best_test_full_ndcg@10           best_valid_full_mrr            \
                                 mean       std                mean       std   
bucket_variant                                                                  
b8                           0.549708       NaN            0.378457       NaN   
baseline                     0.503589  0.060311            0.364176  0.011134   

                ... best_test_sampled_ndcg@5            \
                ...                     mean       std   
bucket_variant  ...                                      
b8              ...                 0.591895       NaN   
baseline        ...                 0.810507  0.000913   

               best_valid_sampled_ndcg@10           best_test_sampled_ndcg@10  \
                                     mean       std                      mean   
bucket_variant                                                                  
b8                               0.558799       NaN                  0.614605   
baseline                         0.559976  0.013212                  0.821241   

                         best_valid_sampled_mrr            \
                     std                   mean       std   
bucket_variant                                              
b8                   NaN               0.559129       NaN   
baseline        0.001001               0.563831  0.008535   

               best_test_sampled_mrr            
                                mean       std  
bucket_variant                                  
b8                          0.616780       NaN  
baseline                    0.816395  0.000301  

[2 rows x 24 columns]

Interpretation guide for NDCG@10:
- first compare `best_valid_full_ndcg@10` and `best_test_full_ndcg@10` across `baseline`, `b8`, and `b9`
- then check whether sampled metrics and MRR show a similar trend
- because baseline is reused from existing runs, only the time-aware runs are newly trained here


## NDCG@5 comparison summary


In [19]:
ndcg5_baseline_runs = [
    'refine_v3_ml50_do025_seed42_ndcg5',
    'refine_v3_ml50_do025_seed2024_ndcg5',
]
ndcg5_timeaware_runs = [
    'timeaware_refine_v3_ml50_do025_seed42_b8_ndcg5',
    'timeaware_refine_v3_ml50_do025_seed2024_b8_ndcg5',
    'timeaware_refine_v3_ml50_do025_seed42_b9_ndcg5',
    'timeaware_refine_v3_ml50_do025_seed2024_b9_ndcg5',
]

baseline_df = rebuild_df(BASELINE_NDCG5_OUTPUT_DIR)
timeaware_df = rebuild_df(TIMEAWARE_NDCG5_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(ndcg5_baseline_runs)].copy()
baseline_subset['model_variant'] = 'baseline'
baseline_subset['bucket_variant'] = 'baseline'

timeaware_subset = timeaware_df[timeaware_df['run_name'].isin(ndcg5_timeaware_runs)].copy()
timeaware_subset['model_variant'] = 'timeaware'
timeaware_subset['bucket_variant'] = timeaware_subset['run_name'].apply(lambda x: 'b8' if '_b8' in x else 'b9')

df_ndcg5 = pd.concat([baseline_subset, timeaware_subset], ignore_index=True)
df_ndcg5 = df_ndcg5.sort_values(['bucket_variant', 'seed', 'run_name']).reset_index(drop=True)
df_ndcg5[[
    'run_name', 'seed', 'bucket_variant', 'use_time_embedding', 'time_bucket_boundaries',
    'best_valid_full_ndcg@5', 'best_test_full_ndcg@5',
    'best_valid_full_ndcg@10', 'best_test_full_ndcg@10',
    'best_valid_full_mrr', 'best_test_full_mrr',
    'best_valid_sampled_ndcg@5', 'best_test_sampled_ndcg@5',
    'best_valid_sampled_ndcg@10', 'best_test_sampled_ndcg@10',
    'best_valid_sampled_mrr', 'best_test_sampled_mrr',
]]


,run_name,seed,bucket_variant,use_time_embedding,time_bucket_boundaries,best_valid_full_ndcg@5,best_test_full_ndcg@5,best_valid_full_ndcg@10,best_test_full_ndcg@10,best_valid_full_mrr,best_test_full_mrr,best_valid_sampled_ndcg@5,best_test_sampled_ndcg@5,best_valid_sampled_ndcg@10,best_test_sampled_ndcg@10,best_valid_sampled_mrr,best_test_sampled_mrr
0,timeaware_refine_v3_ml50_do025_seed2024_b8_ndcg5,2024,b8,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0",0.343342,0.456528,0.504194,0.534630,0.365027,0.393623,0.538029,0.516153,0.566497,0.550565,0.562444,0.548457
1,timeaware_refine_v3_ml50_do025_seed42_b9_ndcg5,42,b9,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0,,,6,0,4,8,0,0",0.418780,0.434443,0.531580,0.550074,0.391962,0.411806,0.558631,0.641690,0.575569,0.684092,0.574527,0.652048
2,timeaware_refine_v3_ml50_do025_seed2024_b9_ndcg5,2024,b9,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0,,,6,0,4,8,0,0",0.334544,0.401456,0.496344,0.476183,0.355911,0.315984,0.559936,0.602599,0.585103,0.624628,0.580096,0.625630
3,refine_v3_ml50_do025_seed42_ndcg5,42,baseline,False,,0.387751,0.416791,0.512117,0.546236,0.372049,0.406292,0.541576,0.809862,0.550634,0.820534,0.557796,0.816182
4,refine_v3_ml50_do025_seed2024_ndcg5,2024,baseline,False,,0.376172,0.337719,0.500475,0.460943,0.356303,0.299817,0.553469,0.811153,0.569319,0.821949,0.569866,0.816608


In [20]:
summary_ndcg5 = df_ndcg5.groupby('bucket_variant')[[
    'best_valid_full_ndcg@5', 'best_test_full_ndcg@5',
    'best_valid_full_ndcg@10', 'best_test_full_ndcg@10',
    'best_valid_full_mrr', 'best_test_full_mrr',
    'best_valid_sampled_ndcg@5', 'best_test_sampled_ndcg@5',
    'best_valid_sampled_ndcg@10', 'best_test_sampled_ndcg@10',
    'best_valid_sampled_mrr', 'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary_ndcg5


best_valid_full_ndcg@5           best_test_full_ndcg@5  \
                                 mean       std                  mean   
bucket_variant                                                          
b8                           0.343342       NaN              0.456528   
b9                           0.376662  0.059564              0.417949   
baseline                     0.381962  0.008188              0.377255   

                         best_valid_full_ndcg@10            \
                     std                    mean       std   
bucket_variant                                               
b8                   NaN                0.504194       NaN   
b9              0.023326                0.513962  0.024916   
baseline        0.055912                0.506296  0.008232   

               best_test_full_ndcg@10           best_valid_full_mrr            \
                                 mean       std                mean       std   
bucket_variant                                                                  
b8                           0.534630       NaN            0.365027       NaN   
b9                           0.513128  0.052249            0.373936  0.025492   
baseline                     0.503589  0.060311            0.364176  0.011134   

                ... best_test_sampled_ndcg@5            \
                ...                     mean       std   
bucket_variant  ...                                      
b8              ...                 0.516153       NaN   
b9              ...                 0.622144  0.027641   
baseline        ...                 0.810507  0.000913   

               best_valid_sampled_ndcg@10           best_test_sampled_ndcg@10  \
                                     mean       std                      mean   
bucket_variant                                                                  
b8                               0.566497       NaN                  0.550565   
b9                               0.580336  0.006741                  0.654360   
baseline                         0.559976  0.013212                  0.821241   

                         best_valid_sampled_mrr            \
                     std                   mean       std   
bucket_variant                                              
b8                   NaN               0.562444       NaN   
b9              0.042047               0.577312  0.003938   
baseline        0.001001               0.563831  0.008535   

               best_test_sampled_mrr            
                                mean       std  
bucket_variant                                  
b8                          0.548457       NaN  
b9                          0.638839  0.018680  
baseline                    0.816395  0.000301  

[3 rows x 24 columns]

Interpretation guide for NDCG@5:
- first compare `best_valid_full_ndcg@5` and `best_test_full_ndcg@5` across `baseline`, `b8`, and `b9`
- then check whether sampled metrics and MRR show a similar trend
- because baseline is reused from existing runs, only the time-aware runs are newly trained here
